# **Cell 1: SETUP**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJ = '/content/drive/MyDrive/neurotwin'
for d in ['data/parc', 'checkpoints']:
    os.makedirs(f'{PROJ}/{d}', exist_ok=True)
!pip -q install nilearn

Mounted at /content/drive
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
# Upgrade bottleneck to resolve the pandas compatibility warning
!pip install --upgrade bottleneck

  Attempting uninstall: bottleneck
    Found existing installation: Bottleneck 1.4.2
    Uninstalling Bottleneck-1.4.2:
      Successfully uninstalled Bottleneck-1.4.2


# **Cell 2: DOWNLOAD DATASETS**

In [ ]:
from nilearn import datasets
dev = datasets.fetch_development_fmri(n_subjects=5, data_dir='/content/nilearn_data')
print(len(dev.func))
print(list(dev.phenotypic.columns))   # shows the columns (age, group, etc.)

[fetch_development_fmri] Added README.md to /content/nilearn_data

[fetch_development_fmri] Dataset created in /content/nilearn_data/development_fmri

[fetch_development_fmri] Downloading data from https://osf.io/yr3av/download ...

[fetch_development_fmri]  ...done. (2 seconds, 0 min)

[fetch_development_fmri] Downloading data from https://osf.io/download/5c8ff3df4712b400183b7092/ ...

# **Cell 3: Inspect shape**

In [ ]:
import nibabel as nib
from nilearn import datasets

# Fetch the dataset if 'dev' is not defined in the workspace
if 'dev' not in globals():
    dev = datasets.fetch_development_fmri(n_subjects=5, data_dir='/content/nilearn_data')

img = nib.load(dev.func[0])
print("image shape:", img.shape)          # (x, y, z, time points)
print("TR per subject:", [nib.load(f).shape[-1] for f in dev.func])
print(dev.phenotypic[['participant_id', 'Age', 'Child_Adult']])

# **Cell 4: Parcellate**

In [ ]:
import numpy as np, json
from nilearn import datasets
from nilearn.maskers import NiftiLabelsMasker

atlas = datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7, resolution_mm=2)
masker = NiftiLabelsMasker(atlas.maps, t_r=2.0, standardize=False, detrend=False)

ids = []
for f, c in zip(dev.func, dev.confounds):
    sid = os.path.basename(f).split('_')[0]        # e.g. sub-pixar001
    out = f'{PROJ}/data/parc/{sid}.npy'
    if not os.path.exists(out):                    # skip if already done
        ts = masker.fit_transform(f, confounds=c)  # shape: (time, 100)
        np.save(out, ts.astype('float32'))
    ids.append(sid)
json.dump(ids, open(f'{PROJ}/data/subject_ids.json', 'w'))

# **Cell 5: Stack into one array and sanity-check**

In [ ]:
series = [np.load(f'{PROJ}/data/parc/{s}.npy') for s in ids]
print([s.shape[0] for s in series])                # time points per subject
Tmin = min(s.shape[0] for s in series)
X = np.stack([s[:Tmin] for s in series])           # (subjects, time, 100)
print("shape:", X.shape)
print("NaNs:", np.isnan(X).any())
print("regions with zero variance:", (X.std(axis=1) == 0).sum())
np.save(f'{PROJ}/data/X_parcellated.npy', X)

In [ ]:
import pandas as pd
pd.DataFrame(dev.phenotypic).to_csv(f'{PROJ}/data/phenotypic.csv', index=False)

# **Cell 6: Check disk space, then download**

In [ ]:
import time
from nilearn import datasets

for attempt in range(1, 21):
    try:
        dev = datasets.fetch_development_fmri(n_subjects=155, data_dir='/content/nilearn_data')
        break
    except Exception as e:
        print(f"attempt {attempt} failed: {type(e).__name__}. Retrying in 15 s...")
        time.sleep(15)
else:
    raise RuntimeError("Download failed after 20 attempts. Do NOT run the later cells.")

assert len(dev.func) == 155, f"Expected 155 subjects, got {len(dev.func)}"
print("downloaded:", len(dev.func))

[fetch_development_fmri] Dataset directory found: /content/nilearn_data/development_fmri

[fetch_development_fmri] Downloading data from https://osf.io/download/5c8ff3f14712b4001a3b560e/ ...

[fetch_development_fmri]  ...done. (4 seconds, 0 min)

[fetch_development_fmri] Downloading data from https://osf.io/download/5c8ff3f1a743a90017608164/ ...

[fetch_development_fmri]  ...done. (8 seconds, 0 min)

[fetch_development_fmri] Downloading data from https://osf.io/download/5c8ff3f12286e80016c3c37e/ ...

... (626 outputs left out)

[fetch_development_fmri]  ...done. (2 seconds, 0 min)

[fetch_development_fmri] Downloading data from https://osf.io/download/5c8ff3de4712b4001a3b55f4/ ...

[fetch_development_fmri]  ...done. (9 seconds, 0 min)

[fetch_development_fmri] Downloading data from https://osf.io/download/5cb47017f2be3c0017056f8d/ ...

# **Cell 7: Parcellate everyone (resumable, with error handling**

In [ ]:
atlas = datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7, resolution_mm=2)
masker = NiftiLabelsMasker(atlas.maps, standardize=False, detrend=False)

ids, failed = [], []
for i, (f, c) in enumerate(zip(dev.func, dev.confounds)):
    sid = os.path.basename(f).split('_')[0]
    out = f'{PROJ}/data/parc/{sid}.npy'
    try:
        if not os.path.exists(out):
            ts = masker.fit_transform(f, confounds=c)
            np.save(out, ts.astype('float32'))
        ids.append(sid)
    except Exception as e:
        failed.append((sid, str(e)[:100]))
    if (i + 1) % 20 == 0:
        print(f"{i+1}/{len(dev.func)} done")

json.dump(ids, open(f'{PROJ}/data/subject_ids.json', 'w'))
print("processed:", len(ids), "| failed:", failed)

# **Cell 8: Stack, check, and save with aligned phenotypes**

In [ ]:
series = [np.load(f'{PROJ}/data/parc/{s}.npy') for s in ids]
lengths = [s.shape[0] for s in series]
print("time points: min", min(lengths), "max", max(lengths))

Tmin = min(lengths)
X = np.stack([s[:Tmin] for s in series]).astype('float32')
print("X shape:", X.shape)
print("NaNs:", np.isnan(X).any(), "| zero-variance regions:", int((X.std(axis=1) == 0).sum()))

np.save(f'{PROJ}/data/X_parcellated.npy', X)
pheno = dev.phenotypic.set_index('participant_id').loc[ids].reset_index()
pheno.to_csv(f'{PROJ}/data/phenotypic.csv', index=False)
print(pheno['Child_Adult'].value_counts())

In [ ]:
import nilearn
print(nilearn.__version__)

# **Cell 9: Motion Check**

In [ ]:
import pandas as pd
c0 = pd.read_csv(dev.confounds[0], sep='\\t')
print(list(c0.columns))

# **Cell 10: look at motion across all 155 subjects**

In [ ]:
fd_mean, fd_frac = {}, {}
for f, c in zip(dev.func, dev.confounds):
    sid = os.path.basename(f).split('_')[0]
    d = pd.read_csv(c, sep='\t')['framewise_displacement'].dropna()
    fd_mean[sid] = d.mean()
    fd_frac[sid] = (d > 0.5).mean()      # fraction of volumes with >0.5 mm jump

motion = pd.DataFrame({'mean_fd': pd.Series(fd_mean), 'frac_over_0.5mm': pd.Series(fd_frac)})
motion = motion.join(pheno.set_index('participant_id')[['Child_Adult', 'Age']])

print(motion['mean_fd'].describe())
print(motion.groupby('Child_Adult')[['mean_fd', 'frac_over_0.5mm']].describe().T)
print(motion.sort_values('mean_fd').tail(10))

motion.to_csv(f'{PROJ}/data/motion.csv')

# **Cell 11 : the counting cell**

In [ ]:
for name, mask in {
    "mean_fd > 0.5":        motion['mean_fd'] > 0.5,
    "mean_fd > 0.7":        motion['mean_fd'] > 0.7,
    ">20% volumes > 0.5mm": motion['frac_over_0.5mm'] > 0.20,
}.items():
    print(name, "-> excluded:", int(mask.sum()),
          "| by group:", motion[mask]['Child_Adult'].value_counts().to_dict(),
          "| remaining:", int((~mask).sum()))

# **Cell 12 : the apply cell**

In [ ]:
RULE = motion['mean_fd'] > 0.5            # replace if chose another rule

keep_ids = [s for s in ids if not RULE[s]]
keep_idx = [ids.index(s) for s in keep_ids]

X_keep = X[keep_idx]
pheno_keep = pheno.set_index('participant_id').loc[keep_ids].reset_index()

np.save(f'{PROJ}/data/X_kept.npy', X_keep)
pheno_keep.to_csv(f'{PROJ}/data/phenotypic_kept.csv', index=False)
import json
json.dump(keep_ids, open(f'{PROJ}/data/subject_ids_kept.json', 'w'))
print(X_keep.shape)
print(pheno_keep['Child_Adult'].value_counts())

# **Cell 13: Split Cell**

In [ ]:
from sklearn.model_selection import train_test_split
import json, numpy as np, pandas as pd

pk = pd.read_csv(f'{PROJ}/data/phenotypic_kept.csv')
ids_k = pk['participant_id'].tolist()
g = dict(zip(pk['participant_id'], pk['Child_Adult']))

# Total available subjects = 35. We must adjust sizes to be valid.
# Let's allocate:
# - test: 10 subjects (~28% of data)
# - cal: 8 subjects (~23% of data)
# - val: 8 subjects (~23% of data)
# - train: remaining 9 subjects (~26% of data)
SEED = 0
rest, test = train_test_split(ids_k, test_size=10, stratify=[g[s] for s in ids_k], random_state=SEED)
rest, cal  = train_test_split(rest,  test_size=8,  stratify=[g[s] for s in rest],  random_state=SEED)
train, val = train_test_split(rest,  test_size=8,  stratify=[g[s] for s in rest],  random_state=SEED)

# nested training subsets, one set per seed (10% inside 25% inside 50% inside 100%)
subsets = {}
for seed in range(5):
    order = np.random.default_rng(seed).permutation(train).tolist()
    subsets[str(seed)] = {str(f): order[:round(f * len(train))] for f in (0.10, 0.25, 0.50, 1.0)}

splits = {'train': train, 'val': val, 'cal': cal, 'test': test, 'subsets': subsets}
json.dump(splits, open(f'{PROJ}/data/splits.json', 'w'))

# checks
parts = [set(train), set(val), set(cal), set(test)]
assert sum(len(p) for p in parts) == len(set().union(*parts)) == len(ids_k), "overlap or missing subjects"
for name in ['train', 'val', 'cal', 'test']:
    c = pd.Series([g[s] for s in splits[name]]).value_counts().to_dict()
    print(name, len(splits[name]), c)
print({f: len(v) for f, v in subsets['0'].items()})